In [ ]:
import sys
!{sys.executable} -m pip install dill --user
!{sys.executable} -m pip install cupy-cuda12x --user
!{sys.executable} -m pip install cuml --user

In [ ]:
!which conda

In [ ]:
!nvidia-smi

In [ ]:
!conda install -c conda-forge cupy cuml

In [ ]:
import cupy as cp

print("✅ CuPy imported successfully!")

# Get number of GPUs
num_gpus = cp.cuda.runtime.getDeviceCount()
print(f"Number of GPUs detected: {num_gpus}")

# Loop through all GPUs and print their names
for i in range(num_gpus):
    props = cp.cuda.runtime.getDeviceProperties(i)
    print(f"GPU {i}: {props['name']}")

In [ ]:
# Run this first to check GPU availability
try:
    import cupy as cp
    import cuml
    print("✅ GPU libraries available!")
    print(f"GPU: {cp.cuda.Device().name}")
except ImportError:
    print("❌ Need to install: cupy-cuda11x cuml")

In [ ]:
# GPU-Accelerated Monte Carlo k-NN Optimization
import numpy as np
import cupy as cp  # GPU arrays
from cuml.neighbors import NearestNeighbors as cuNearestNeighbors  # GPU k-NN
from scipy.sparse.csgraph import connected_components
import random
from tqdm import tqdm
import time

class GPUMonteCarloKNNOptimizer:
    def __init__(self, data_root='${TDL_ROOT_DIR}/results/D1', true_b0=10):
        self.data_root = data_root
        self.true_b0 = true_b0
        self.full_dataset = None
        self.gpu_dataset = None
        
        # Check GPU availability
        try:
            import cupy as cp
            import cuml
            self.gpu_available = True
            print("✅ GPU acceleration available")
            print(f"   GPU: {cp.cuda.Device().name}")
            print(f"   Memory: {cp.cuda.MemInfo().total / 1e9:.1f} GB")
        except ImportError:
            self.gpu_available = False
            print("❌ GPU libraries not available. Install cupy and cuml for acceleration.")
    
    def load_gpu_data(self, cpu_data):
        """Transfer data to GPU memory"""
        if self.gpu_available:
            self.gpu_dataset = cp.asarray(cpu_data, dtype=cp.float32)
            print(f"📤 Data transferred to GPU: {self.gpu_dataset.shape}")
        else:
            self.gpu_dataset = cpu_data
    
    def find_k_for_subset_gpu(self, subset_indices, max_k=100):
        """GPU-accelerated k-finding for a subset"""
        if not self.gpu_available:
            return self.find_k_for_subset_cpu(subset_indices, max_k)
        
        # Extract subset on GPU
        subset_gpu = self.gpu_dataset[subset_indices]
        subset_size = len(subset_indices)
        
        for k in range(1, min(max_k, subset_size)):
            try:
                # GPU k-NN computation
                neighbors = cuNearestNeighbors(n_neighbors=k+1)
                neighbors.fit(subset_gpu)
                
                # Get connectivity matrix
                distances, indices = neighbors.kneighbors(subset_gpu)
                
                # Create adjacency matrix on GPU
                adjacency = cp.zeros((subset_size, subset_size), dtype=cp.int32)
                for i in range(subset_size):
                    for j in range(1, k+1):  # Skip self (index 0)
                        neighbor_idx = indices[i, j]
                        adjacency[i, neighbor_idx] = 1
                        adjacency[neighbor_idx, i] = 1  # Make symmetric
                
                # Transfer to CPU for connected components (scipy is CPU-only)
                adjacency_cpu = cp.asnumpy(adjacency)
                n_components, _ = connected_components(adjacency_cpu, directed=False)
                
                if n_components == self.true_b0:
                    return k, subset_indices
                    
            except Exception as e:
                continue
        
        return None, None
    
    def find_k_for_subset_cpu(self, subset_indices, max_k=100):
        """CPU fallback version"""
        subset = self.full_dataset[subset_indices]
        
        for k in range(1, min(max_k, len(subset))):
            try:
                from sklearn.neighbors import NearestNeighbors
                neighbors = NearestNeighbors(n_neighbors=k+1).fit(subset)
                graph = neighbors.kneighbors_graph(subset, mode='connectivity')
                
                graph.setdiag(0)
                graph_undirected = graph.maximum(graph.T)
                graph_undirected.eliminate_zeros()
                
                n_components, _ = connected_components(csgraph=graph_undirected, directed=False)
                
                if n_components == self.true_b0:
                    return k, subset_indices
                    
            except Exception as e:
                continue
        
        return None, None
    
    def parallel_monte_carlo_gpu(self, full_dataset, n_trials=1000, subset_fraction=0.25, max_k=100, batch_size=50):
        """GPU-accelerated Monte Carlo with batching"""
        self.full_dataset = full_dataset
        self.load_gpu_data(full_dataset)
        
        total_points = len(full_dataset)
        subset_size = int(total_points * subset_fraction)
        
        print(f"🚀 GPU Monte Carlo Optimization:")
        print(f"  Trials: {n_trials}")
        print(f"  Subset size: {subset_size} ({subset_fraction*100}%)")
        print(f"  Batch size: {batch_size}")
        print(f"  Max k: {max_k}")
        
        successful_results = []  # Store (k, subset_indices) pairs
        
        # Process in batches for memory efficiency
        for batch_start in tqdm(range(0, n_trials, batch_size), desc="GPU batches"):
            batch_end = min(batch_start + batch_size, n_trials)
            batch_results = []
            
            # Generate batch of random subsets
            batch_subsets = []
            for _ in range(batch_start, batch_end):
                subset_indices = np.random.choice(total_points, subset_size, replace=False)
                batch_subsets.append(subset_indices)
            
            # Process batch (can be parallelized further)
            for subset_indices in batch_subsets:
                result = self.find_k_for_subset_gpu(subset_indices, max_k)
                if result[0] is not None:
                    batch_results.append(result)
            
            successful_results.extend(batch_results)
        
        successful_ks = [result[0] for result in successful_results]
        successful_subsets = [result[1] for result in successful_results]
        
        print(f"\n🎯 GPU Results:")
        print(f"  Successful trials: {len(successful_results)}")
        print(f"  Success rate: {len(successful_results)/n_trials*100:.1f}%")
        
        return successful_ks, successful_subsets

# Alternative: Multi-threading CPU acceleration (if no GPU)
import concurrent.futures
from multiprocessing import Pool

class MultiprocessingMonteCarloOptimizer:
    def __init__(self, true_b0=10):
        self.true_b0 = true_b0
    
    @staticmethod
    def worker_find_k(args):
        """Worker function for multiprocessing"""
        full_dataset, subset_indices, max_k, true_b0 = args
        subset = full_dataset[subset_indices]
        
        for k in range(1, min(max_k, len(subset))):
            try:
                from sklearn.neighbors import NearestNeighbors
                neighbors = NearestNeighbors(n_neighbors=k+1).fit(subset)
                graph = neighbors.kneighbors_graph(subset, mode='connectivity')
                
                graph.setdiag(0)
                graph_undirected = graph.maximum(graph.T)
                graph_undirected.eliminate_zeros()
                
                n_components, _ = connected_components(csgraph=graph_undirected, directed=False)
                
                if n_components == true_b0:
                    return k, subset_indices
                    
            except Exception:
                continue
        
        return None, None
    
    def parallel_monte_carlo_cpu(self, full_dataset, n_trials=1000, subset_fraction=0.25, max_k=100, n_processes=8):
        """CPU multiprocessing acceleration"""
        total_points = len(full_dataset)
        subset_size = int(total_points * subset_fraction)
        
        print(f"🔄 CPU Multiprocessing Monte Carlo:")
        print(f"  Processes: {n_processes}")
        print(f"  Trials: {n_trials}")
        
        # Generate all random subsets
        tasks = []
        for _ in range(n_trials):
            subset_indices = np.random.choice(total_points, subset_size, replace=False)
            tasks.append((full_dataset, subset_indices, max_k, self.true_b0))
        
        # Process in parallel
        successful_results = []
        
        with Pool(processes=n_processes) as pool:
            results = list(tqdm(
                pool.imap(self.worker_find_k, tasks),
                total=n_trials,
                desc="CPU parallel"
            ))
        
        successful_results = [r for r in results if r[0] is not None]
        successful_ks = [result[0] for result in successful_results]
        successful_subsets = [result[1] for result in successful_results]
        
        print(f"\n🎯 CPU Parallel Results:")
        print(f"  Successful trials: {len(successful_results)}")
        print(f"  Success rate: {len(successful_results)/n_trials*100:.1f}%")
        
        return successful_ks, successful_subsets

In [ ]:

# Load your data first with the original optimizer
optimizer = MonteCarloKNNOptimizer()
optimizer.load_data()
full_data = optimizer.extract_data_points(optimizer.train_data)

# Now use GPU version
gpu_optimizer = GPUMonteCarloKNNOptimizer(true_b0=10)
successful_ks, successful_subsets, successful_indices = gpu_optimizer.parallel_monte_carlo_gpu(
    full_data, 
    n_trials=1000, 
    batch_size=100
)

print(f"🎯 Results: Found {len(successful_ks)} successful trials!")
print(f"Sample k values: {successful_ks[:10]}")
print(f"First successful subset shape: {successful_subsets[0].shape}")